# RAG 파라미터 실험 노트북

**구조**: 가설 → 실험 → 관찰 → 결론

## 실험 목록

| 번호 | 실험 | 변수 | 범위 |
|------|------|------|------|
| 1 | Chunk Size 변화 | 200 / 500 / 1000 | ✅ 범위 내 |
| 2 | Top-k 변경 | 3 / 5 / 10 | ✅ 범위 내 |
| 3 | Temperature 변화 | 0 / 0.3 / 0.7 | ✅ 범위 내 |
| 4-a | FAISS vs Chroma | DB 종류 | ✅ 범위 내 |
| 4-b | Index Type (Flat vs IVF) | FAISS 인덱스 | ⚠️ IVF는 강의 범위 초과 (코드 포함) |
| 4-c | 검색 속도 측정 | - | ✅ 범위 내 |
| 5 | 실패 케이스 분석 | - | ✅ 범위 내 |
| 6-a | System Prompt 강화 | 프롬프트 | ✅ 범위 내 |
| 6-b | 질문 재작성 | Query Rewriting | ✅ 범위 내 |
| 6-c | Re-ranking | Reranker 모델 | ❌ 범위 초과 (개념 + 참고 코드만) |

> **데이터**: `data/테슬라_KR.txt`, `data/리비안_KR.txt`  
> **LLM**: `gpt-4.1-mini` (실험용 경량 모델)  
> **Embedding**: `text-embedding-3-small`

In [ ]:
# ──────────────────────────────────────
# 공통 설정 (모든 실험에서 공유)
# ──────────────────────────────────────
import os
import time
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

try:
    import faiss
    print("✅ faiss 로드 성공")
except ImportError:
    print("⚠️ faiss 미설치: pip install faiss-cpu 실행 필요 (실험 4-b 제외하고 진행 가능)")

# 공유 임베딩 모델 (API 호출 절약)
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 기본 LLM (실험별로 temperature 등 조정)
DEFAULT_MODEL = "gpt-4.1-mini"
llm = ChatOpenAI(model=DEFAULT_MODEL, temperature=0)

print(f"✅ 설정 완료 | 모델: {DEFAULT_MODEL}")

In [ ]:
# ──────────────────────────────────────
# 문서 로드
# ──────────────────────────────────────
loaders = [
    TextLoader("data/테슬라_KR.txt", encoding="utf-8"),
    TextLoader("data/리비안_KR.txt", encoding="utf-8"),
]
raw_docs = []
for loader in loaders:
    raw_docs.extend(loader.load())

print(f"✅ 로드된 문서 수: {len(raw_docs)}")
for doc in raw_docs:
    src = doc.metadata.get("source", "unknown")
    print(f"  - {src}: {len(doc.page_content):,} 글자")

In [ ]:
# ──────────────────────────────────────
# 공통 헬퍼 함수
# ──────────────────────────────────────

def build_chroma(docs, chunk_size=500, chunk_overlap=50, collection_name="exp"):
    """청크 설정으로 Chroma 벡터스토어 생성."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    split_docs = splitter.split_documents(docs)

    # 기존 컬렉션 삭제 후 재생성 (재실험 충돌 방지)
    try:
        Chroma(collection_name=collection_name,
               embedding_function=embedding_model).delete_collection()
    except Exception:
        pass

    vs = Chroma.from_documents(
        documents=split_docs,
        embedding=embedding_model,
        collection_name=collection_name
    )
    return vs, split_docs


def build_faiss(docs, chunk_size=500, chunk_overlap=50):
    """FAISS 벡터스토어 생성."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    split_docs = splitter.split_documents(docs)
    vs = FAISS.from_documents(documents=split_docs, embedding=embedding_model)
    return vs, split_docs


def build_rag_chain(vectorstore, llm, k=3, system_prompt=None):
    """RAG 체인 구성."""
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    if system_prompt is None:
        system_prompt = (
            "당신은 도움이 되는 AI 어시스턴트입니다. "
            "반드시 아래 컨텍스트를 기반으로 답하세요. "
            "컨텍스트에 없으면 '제공된 문서에서 찾을 수 없습니다'라고 하세요.\n\n"
            "컨텍스트:\n{context}"
        )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}"),
    ])

    def format_docs(docs):
        return "\n\n---\n\n".join(doc.page_content for doc in docs)

    chain = (
        RunnableParallel({
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        })
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def run_questions(chain, questions):
    """질문 리스트 실행 → 결과 딕셔너리 리스트 반환."""
    results = []
    for q in questions:
        t0 = time.time()
        response = chain.invoke(q)
        elapsed = time.time() - t0
        results.append({
            "질문": q,
            "응답": response,
            "응답_길이": len(response),
            "응답_시간(s)": round(elapsed, 2),
        })
    return results


print("✅ 헬퍼 함수 정의 완료")

In [ ]:
# ──────────────────────────────────────
# 공통 테스트 질문 (모든 실험에서 동일 사용)
# ──────────────────────────────────────
TEST_QUESTIONS = [
    "테슬라의 주요 전기차 모델을 소개해주세요.",
    "리비안은 어떤 회사이며, 주요 제품은 무엇인가요?",
    "테슬라와 리비안의 주요 차이점은 무엇인가요?",
]

print("✅ 테스트 질문:")
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  Q{i}: {q}")

---
## 실험 1: Chunk Size 변화

### 가설
> - **chunk_size 小**: 정밀한 검색 가능하지만, 필요한 맥락이 여러 청크에 분산될 수 있다.  
> - **chunk_size 大**: 더 많은 맥락이 포함되지만, 관련 없는 내용이 섞여 검색 정밀도가 낮아질 수 있다.

### 실험 설계
| 파라미터 | 값 |
|---------|----||
| **변수** | `chunk_size` = 200, 500, 1000 |
| chunk_overlap | chunk_size × 10% |
| k | 3 (고정) |
| temperature | 0 (고정) |

### 📚 참고 자료
| 파일 | 관련 섹션 |
|------|----------|
| `PRJ01_W2_004_Text_Splitter.ipynb` | `텍스트 분할(Text Splitting)` › `### 1. CharacterTextSplitter`, `### 2. RecursiveCharacterTextSplitter` |
| `PRJ01_W2_004_Text_Splitter.ipynb` | `### chunk_size와 chunk_overlap 설정 가이드` |
| `PRJ01_W2_002_Simple_RAG_Pipeline.ipynb` | `### Step 1: Indexing` › `` `2. 문서 청크 분할(Split Texts)` `` |

In [ ]:
chunk_sizes = [200, 500, 1000]
exp1_results = {}

for cs in chunk_sizes:
    overlap = int(cs * 0.1)
    print(f"\n🔧 chunk_size={cs}, overlap={overlap} 실험 중...")

    vs, split_docs = build_chroma(
        raw_docs,
        chunk_size=cs,
        chunk_overlap=overlap,
        collection_name=f"exp1_cs{cs}"
    )
    chain = build_rag_chain(vs, llm, k=3)
    results = run_questions(chain, TEST_QUESTIONS)
    exp1_results[cs] = {"results": results, "n_chunks": len(split_docs)}

    avg_len = sum(r["응답_길이"] for r in results) / len(results)
    print(f"  청크 수: {len(split_docs)} | 평균 응답 길이: {avg_len:.0f}자")

print("\n✅ 실험 1 완료")

In [ ]:
# 결과 요약 테이블
rows = []
for cs, data in exp1_results.items():
    results = data["results"]
    rows.append({
        "chunk_size": cs,
        "생성된 청크 수": data["n_chunks"],
        "평균 응답 길이(자)": round(sum(r["응답_길이"] for r in results) / len(results)),
        "평균 응답 시간(s)": round(sum(r["응답_시간(s)"] for r in results) / len(results), 2),
    })

df1 = pd.DataFrame(rows)
print("=== 실험 1 요약 ===")
display(df1)

# Q3(비교 질문) 응답 상세 비교
print("\n=== Q3 응답 비교 (테슬라 vs 리비안 차이점) ===")
for cs, data in exp1_results.items():
    print(f"\n[chunk_size={cs}]")
    print(data["results"][2]["응답"][:400], "...")

### 관찰 및 결론

| chunk_size | 청크 수 | 응답 정확도 | 맥락 유지 | 특이사항 |
|------------|--------|------------|----------|--------|
| 200 | | | | |
| 500 | | | | |
| 1000 | | | | |

**결론**: *(실행 후 직접 작성)*

---

## 실험 2: Top-k 변경

### 가설
> - **k 小**: 검색 노이즈 적고 응답이 정밀하지만 중요한 정보 누락 가능성.  
> - **k 大**: 더 많은 정보가 컨텍스트에 포함되지만, 관련 없는 청크가 섞여 응답 품질 저하 가능성.

### 실험 설계
| 파라미터 | 값 |
|---------|----||
| **변수** | `k` = 3, 5, 10 |
| chunk_size | 500 (고정) |
| temperature | 0 (고정) |

### 📚 참고 자료
| 파일 | 관련 섹션 |
|------|----------|
| `PRJ01_W2_007_Retriever.ipynb` | `# 벡터 저장소 기반 RAG 검색기 (Retriever)` › `` `(2) Top K` ``, `` `(3) 임계값 지정` ``, `` `(4) MMR(Maximal Marginal Relevance) 검색` `` |
| `PRJ01_W2_002_Simple_RAG_Pipeline.ipynb` | `### Step 2: Retrieval and generation` › `` `5. 검색 및 생성` `` |

In [ ]:
# chunk_size=500 벡터스토어 생성
vs_exp2, _ = build_chroma(raw_docs, chunk_size=500, collection_name="exp2_topk")

k_values = [3, 5, 10]
exp2_results = {}

for k in k_values:
    print(f"\n🔧 k={k} 실험 중...")
    chain = build_rag_chain(vs_exp2, llm, k=k)
    results = run_questions(chain, TEST_QUESTIONS)
    exp2_results[k] = results

    avg_len = sum(r["응답_길이"] for r in results) / len(results)
    print(f"  평균 응답 길이: {avg_len:.0f}자")

print("\n✅ 실험 2 완료")

In [ ]:
rows2 = []
for k, results in exp2_results.items():
    rows2.append({
        "k (top-k)": k,
        "컨텍스트 청크 수": k,
        "평균 응답 길이(자)": round(sum(r["응답_길이"] for r in results) / len(results)),
        "평균 응답 시간(s)": round(sum(r["응답_시간(s)"] for r in results) / len(results), 2),
    })

df2 = pd.DataFrame(rows2)
print("=== 실험 2 요약 ===")
display(df2)

# Q3 응답 비교 (비교 질문은 더 많은 청크가 필요)
print("\n=== Q3 응답 비교 (비교 질문) ===")
for k, results in exp2_results.items():
    print(f"\n[k={k}]")
    print(results[2]["응답"][:400], "...")

### 관찰 및 결론

| k | 응답 포괄성 | 노이즈 영향 | 응답 시간 |
|---|------------|------------|----------|
| 3 | | | |
| 5 | | | |
| 10 | | | |

**결론**: *(실행 후 직접 작성)*

---

## 실험 3: Temperature 변화

### 가설
> - **temperature=0**: 결정론적 응답, 사실 기반 RAG에 적합하지만 단조로울 수 있음.  
> - **temperature 높음**: 다양한 표현이 가능하나, 컨텍스트를 벗어난 hallucination 가능성 증가.  
> - **RAG에서는 낮은 temperature가 유리**: 컨텍스트 이탈 억제.

### 실험 설계
| 파라미터 | 값 |
|---------|----||
| **변수** | `temperature` = 0, 0.3, 0.7 |
| chunk_size | 500 (고정) |
| k | 3 (고정) |
| 반복 횟수 | 3회 (일관성 측정) |

### 📚 참고 자료
| 파일 | 관련 섹션 |
|------|----------|
| `PRJ01_W1_002_OpenAI_Chat_Completion.ipynb` | `## 5. 매개변수 최적화` › `### 5.1 주요 매개변수`, `### 5.2 시나리오별 설정` |
| `PRJ01_W2_007_Retriever.ipynb` | `# [실습 프로젝트] Naive RAG 구현` › `` `(4) RAG 체인 구성` `` (`temperature=0` 적용 사례) |

In [ ]:
vs_exp3, _ = build_chroma(raw_docs, chunk_size=500, collection_name="exp3_temp")

# 일관성 측정용 고정 질문
CONSISTENCY_Q = "테슬라의 주요 전기차 모델을 소개해주세요."

temperatures = [0, 0.3, 0.7]
exp3_results = {}

for temp in temperatures:
    print(f"\n🌡️ temperature={temp} 실험 중...")
    temp_llm = ChatOpenAI(model=DEFAULT_MODEL, temperature=temp)
    chain = build_rag_chain(vs_exp3, temp_llm, k=3)

    # 동일 질문 3회 반복 → 응답 일관성 측정
    responses = [chain.invoke(CONSISTENCY_Q) for _ in range(3)]
    exp3_results[temp] = responses

    lengths = [len(r) for r in responses]
    print(f"  응답 길이 (3회): {lengths}")

print("\n✅ 실험 3 완료")

In [ ]:
rows3 = []
for temp, responses in exp3_results.items():
    lengths = [len(r) for r in responses]
    # 응답 간 차이: 가장 긴 응답과 가장 짧은 응답의 글자 수 차이
    spread = max(lengths) - min(lengths)
    rows3.append({
        "temperature": temp,
        "응답 길이 (1회)": lengths[0],
        "응답 길이 (2회)": lengths[1],
        "응답 길이 (3회)": lengths[2],
        "최대-최소 편차(자)": spread,
    })

df3 = pd.DataFrame(rows3)
print("=== 실험 3 요약 ===")
display(df3)

print("\n=== temperature=0 응답 일관성 확인 ===")
for i, r in enumerate(exp3_results[0], 1):
    print(f"\n[{i}회차]\n{r[:250]}...")

print("\n=== temperature=0.7 응답 일관성 확인 ===")
for i, r in enumerate(exp3_results[0.7], 1):
    print(f"\n[{i}회차]\n{r[:250]}...")

### 관찰 및 결론

| temperature | 응답 일관성(편차) | 표현 다양성 | hallucination 위험 | RAG 적합도 |
|------------|----------------|------------|-------------------|------------|
| 0 | | | | |
| 0.3 | | | | |
| 0.7 | | | | |

**RAG에서의 temperature 권장값**: *(실행 후 작성)*

**결론**: *(실행 후 직접 작성)*

---

## 실험 4: 벡터DB 구조 실험

### 가설
> - **FAISS**: 인메모리, 빠른 검색 속도, 소~중규모 프로토타이핑에 유리.  
> - **Chroma**: 영속성 지원, 메타데이터 필터링 강력, 관리 기능 풍부.  
> - **인덱스 타입**: FlatL2(정확)는 소규모, IVF(근사)는 대규모에 유리한 trade-off.

### 📚 참고 자료
| 파일 | 관련 섹션 |
|------|----------|
| `PRJ01_W2_006_Vectorstore.ipynb` | `# 벡터 저장소 (Vector Store)` › `### 1. Chroma`, `### 2. FAISS(Facebook AI Similarity Search)` |

### 4-a: FAISS vs Chroma 비교

> 📚 `PRJ01_W2_006_Vectorstore.ipynb` — `### 1. Chroma` › `` `(1) 벡터 저장소 초기화` `` ~ `` `(4) 벡터 저장소 로드` `` / `### 2. FAISS` › `` `(1)` `` ~ `` `(4) 로컬에 저장 및 로드` ``

In [ ]:
# 동일 데이터로 FAISS / Chroma 각각 구축 및 구축 시간 측정
splitter_base = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs_base = splitter_base.split_documents(raw_docs)
print(f"공통 청크 수: {len(split_docs_base)}")

# --- Chroma ---
try:
    Chroma(collection_name="exp4_chroma",
           embedding_function=embedding_model).delete_collection()
except Exception:
    pass

t0 = time.time()
chroma_db = Chroma.from_documents(
    documents=split_docs_base,
    embedding=embedding_model,
    collection_name="exp4_chroma"
)
chroma_build = time.time() - t0
print(f"Chroma 구축 시간: {chroma_build:.2f}s")

# --- FAISS (FlatL2) ---
t0 = time.time()
faiss_db = FAISS.from_documents(
    documents=split_docs_base,
    embedding=embedding_model
)
faiss_build = time.time() - t0
print(f"FAISS 구축 시간: {faiss_build:.2f}s")
print(f"FAISS 인덱스 벡터 수: {faiss_db.index.ntotal}")

In [ ]:
# 검색 속도 비교 (N회 반복)
QUERY = "테슬라의 전기차 모델"
N_REPEAT = 20

chroma_times, faiss_times = [], []

for _ in range(N_REPEAT):
    t0 = time.time()
    chroma_db.similarity_search(QUERY, k=3)
    chroma_times.append(time.time() - t0)

for _ in range(N_REPEAT):
    t0 = time.time()
    faiss_db.similarity_search(QUERY, k=3)
    faiss_times.append(time.time() - t0)

df4a = pd.DataFrame({
    "벡터DB": ["Chroma", "FAISS (FlatL2)"],
    "구축 시간(s)": [round(chroma_build, 3), round(faiss_build, 3)],
    f"검색 평균(ms, N={N_REPEAT})": [
        round(sum(chroma_times) / N_REPEAT * 1000, 2),
        round(sum(faiss_times) / N_REPEAT * 1000, 2)
    ],
    "영속성 지원": ["✅ persist_directory", "✅ save_local"],
    "메타데이터 필터": ["✅ 강력", "⚠️ 제한적"],
    "인메모리": ["❌ (디스크 기반)", "✅"],
})
print("=== 4-a: FAISS vs Chroma ===")
display(df4a)

### 4-b: Index Type 변경 (Flat vs IVF)

> ⚠️ **강의 범위 참고**: `PRJ01_W2_006_Vectorstore.ipynb` › `### 2. FAISS` › `` `(1) 벡터 저장소 초기화` `` 에서는 `IndexFlatL2`만 다룸. `IndexIVFFlat`은 해당 노트북에 포함되지 않은 확장 내용.

| 항목 | IndexFlatL2 | IndexIVFFlat |
|------|------------|-------------|
| 검색 방식 | 완전 탐색 (Exact) | 근사 탐색 (Approximate) |
| 속도 | O(n) | O(n / nlist) |
| 정확도 | 100% | nprobe에 따라 조절 |
| 사전 학습 | 불필요 | 필요 (클러스터링) |
| 적합 규모 | < 100K 벡터 | > 100K 벡터 |

In [ ]:
# ⚠️ IndexIVFFlat - 강의 범위 초과, 참고 학습용
try:
    DIMENSION = 1536  # text-embedding-3-small 차원
    N_LIST = 10       # 클러스터 수 (일반적으로 sqrt(n_vectors) 권장)

    # 학습 데이터 준비 (IVF는 사전 클러스터링 학습 필요)
    sample_texts = [doc.page_content for doc in split_docs_base[:60]]
    print("임베딩 생성 중 (IVF 학습용)...")
    sample_vecs = np.array(
        embedding_model.embed_documents(sample_texts), dtype=np.float32
    )

    # Flat 인덱스
    flat_index = faiss.IndexFlatL2(DIMENSION)
    flat_index.add(sample_vecs)

    # IVF 인덱스 (학습 후 추가)
    quantizer = faiss.IndexFlatL2(DIMENSION)
    ivf_index = faiss.IndexIVFFlat(quantizer, DIMENSION, N_LIST)
    ivf_index.train(sample_vecs)  # 클러스터링 학습 필요
    ivf_index.add(sample_vecs)
    ivf_index.nprobe = 3  # 탐색할 클러스터 수 (높을수록 정확하지만 느림)

    # 속도 비교
    query_vec = np.array(
        [embedding_model.embed_query("테슬라 전기차 모델")], dtype=np.float32
    )
    N_BENCH = 200
    flat_times, ivf_times = [], []

    for _ in range(N_BENCH):
        t0 = time.time()
        flat_index.search(query_vec, 3)
        flat_times.append(time.time() - t0)

        t0 = time.time()
        ivf_index.search(query_vec, 3)
        ivf_times.append(time.time() - t0)

    df4b = pd.DataFrame({
        "인덱스 타입": ["IndexFlatL2", f"IndexIVFFlat (nlist={N_LIST}, nprobe=3)"],
        f"평균 검색 시간(μs, N={N_BENCH})": [
            round(sum(flat_times) / N_BENCH * 1e6, 2),
            round(sum(ivf_times) / N_BENCH * 1e6, 2)
        ],
        "검색 방식": ["Exact (정확)", "Approximate (근사)"],
        "추천 규모": ["소규모 (< 100K)", "대규모 (> 100K)"],
    })
    print("=== 4-b: Index Type 비교 ===")
    display(df4b)
    print("\n💡 소규모 데이터셋에서는 IVF 이점이 미미합니다.")
    print("   100만+ 벡터 규모에서 IVF의 속도 이점이 두드러집니다.")

except NameError:
    print("⚠️ faiss 미설치 - 실험 4-b 건너뜀")
    print("   pip install faiss-cpu 후 재실행하세요.")

### 4-c: End-to-End RAG 검색 속도 측정

> 📚 `PRJ01_W2_006_Vectorstore.ipynb` — `### 1. Chroma` / `### 2. FAISS` › `` `(3) 문서 검색` `` (유사도 검색 API)  
> 📚 `PRJ01_W2_007_Retriever.ipynb` — `# [실습 프로젝트] Naive RAG 구현` › `` `(4) RAG 체인 구성` ``

In [ ]:
# End-to-End (검색 + LLM) 속도 비교
speed_questions = [
    "테슬라의 주요 모델은?",
    "리비안의 창업 배경은?",
    "전기차 배터리 기술은?",
    "테슬라 자율주행 기능은?",
    "리비안 주요 투자자는?",
]

chroma_chain = build_rag_chain(chroma_db, llm, k=3)
faiss_chain = build_rag_chain(faiss_db, llm, k=3)

chroma_e2e, faiss_e2e = [], []

print("Chroma E2E 측정 중...")
for q in speed_questions:
    t0 = time.time()
    chroma_chain.invoke(q)
    chroma_e2e.append(time.time() - t0)

print("FAISS E2E 측정 중...")
for q in speed_questions:
    t0 = time.time()
    faiss_chain.invoke(q)
    faiss_e2e.append(time.time() - t0)

df4c = pd.DataFrame({
    "질문": speed_questions,
    "Chroma 응답 시간(s)": [round(t, 2) for t in chroma_e2e],
    "FAISS 응답 시간(s)": [round(t, 2) for t in faiss_e2e],
})
print("\n=== 4-c: End-to-End 속도 비교 ===")
display(df4c)
print(f"\nChroma 평균: {sum(chroma_e2e)/len(chroma_e2e):.2f}s")
print(f"FAISS  평균: {sum(faiss_e2e)/len(faiss_e2e):.2f}s")
print("\n💡 E2E에서는 LLM 응답 시간이 지배적이라 DB 차이가 희석됩니다.")

### 관찰 및 결론

#### 4-a: FAISS vs Chroma
| 항목 | 관찰 내용 |
|------|----------|
| 구축 속도 | |
| 검색 속도 | |
| 사용 편의성 | |

#### 4-b: Index Type (Flat vs IVF)
| 항목 | 관찰 내용 |
|------|----------|
| 속도 차이 | |
| 현재 규모에서의 의미 | |

#### 4-c: 속도 vs 정확도 Trade-off
- 소규모 프로토타입: **FlatL2 + Chroma** (간편, 정확)
- 대규모 프로덕션: **IVF + Pinecone/Weaviate** (속도 우선)

**결론**: *(실행 후 직접 작성)*

---

## 실험 5: "실패 케이스" 분석

### 가설
> RAG 시스템의 오답은 세 가지 원인으로 분류할 수 있다:
> 1. **Chunk 문제**: 필요한 정보가 청크 경계에서 잘려 불완전한 컨텍스트 제공
> 2. **Embedding 의미 왜곡**: 질문과 관련 청크가 벡터 공간에서 멀리 매핑됨
> 3. **LLM Hallucination**: 컨텍스트와 무관한 내용 생성

### 실패 유도 질문 5가지 설계
| # | 질문 | 예상 실패 원인 |
|---|------|---------------|
| 1 | 두 회사의 2024년 매출 합계 계산 | LLM 수치 hallucination |
| 2 | 테슬라 CEO의 SNS 최근 발언 | 문서 범위 초과 |
| 3 | R1T vs 사이버트럭 배터리 용량 비교 | Chunk 분산 (멀티홉) |
| 4 | 전기차 (한 단어 질문) | Embedding 모호성 |
| 5 | 이 회사들의 향후 주가 전망 | LLM 추측성 hallucination |

### 📚 참고 자료
| 파일 | 관련 섹션 |
|------|----------|
| `PRJ01_W2_002_Simple_RAG_Pipeline.ipynb` | `## 2. 최신 LangChain을 활용한 RAG 구현` (전체 파이프라인 — 각 단계의 실패 원인 분석 기반) |
| `PRJ01_W2_007_Retriever.ipynb` | `# [실습 프로젝트] Naive RAG 구현` |
| `PRJ01_W2_005_Embedding_Model.ipynb` | (Embedding 의미 왜곡 원인 이해를 위한 배경 지식) |

In [ ]:
failure_cases = [
    {
        "질문": "테슬라와 리비안의 2024년 전체 매출 합계를 계산해주세요.",
        "예상_원인": "LLM Hallucination (수치 계산)",
        "판단_기준": "구체적 수치를 자신 있게 제시하면 실패",
    },
    {
        "질문": "테슬라 CEO가 최근 소셜미디어에 올린 발언을 요약해주세요.",
        "예상_원인": "문서 범위 초과",
        "판단_기준": "없는 내용을 만들어내면 실패",
    },
    {
        "질문": "리비안 R1T의 배터리 용량과 테슬라 사이버트럭의 배터리 용량 중 어느 것이 더 큰가요?",
        "예상_원인": "Chunk 분산 (두 정보가 다른 청크에 있어 비교 어려움)",
        "판단_기준": "두 정보 중 하나를 모른다고 하거나 잘못 비교하면 실패",
    },
    {
        "질문": "전기차",
        "예상_원인": "Embedding 의미 왜곡 (너무 모호한 질문)",
        "판단_기준": "엉뚱한 청크 검색으로 질문과 무관한 답변 생성",
    },
    {
        "질문": "테슬라와 리비안의 향후 주가 전망은 어떻게 되나요?",
        "예상_원인": "LLM Hallucination (추측/예측 영역)",
        "판단_기준": "예측 수치나 근거 없는 전망을 제시하면 실패",
    },
]

print(f"✅ 실패 케이스 {len(failure_cases)}개 설계 완료")

In [ ]:
vs_exp5, _ = build_chroma(raw_docs, chunk_size=500, collection_name="exp5_failure")
chain_exp5 = build_rag_chain(vs_exp5, llm, k=3)
retriever_exp5 = vs_exp5.as_retriever(search_kwargs={"k": 3})

failure_results = []
for i, case in enumerate(failure_cases, 1):
    print(f"\n{'='*60}")
    print(f"케이스 {i}: {case['질문']}")
    print(f"예상 원인: {case['예상_원인']}")

    response = chain_exp5.invoke(case["질문"])
    retrieved = retriever_exp5.invoke(case["질문"])

    print(f"\n[응답]\n{response[:300]}...")
    print(f"\n[검색된 청크 1번 (처음 100자)]\n{retrieved[0].page_content[:100]}...")

    # 거부 판단: '없습니다', '모릅니다', '찾을 수 없' 등이 있으면 정직한 거부
    refused = any(kw in response for kw in ["없습니다", "모릅니다", "찾을 수 없", "확인되지", "제공되지"])

    failure_results.append({
        "케이스": i,
        "질문": case["질문"],
        "예상_원인": case["예상_원인"],
        "응답": response,
        "정직한_거부": "✅" if refused else "❌ 답변 생성",
    })

print("\n✅ 실험 5 완료")

In [ ]:
df5 = pd.DataFrame([{
    "케이스": r["케이스"],
    "질문 (요약)": r["질문"][:35] + "...",
    "예상 실패 원인": r["예상_원인"],
    "정직한 거부 여부": r["정직한_거부"],
} for r in failure_results])

print("=== 실험 5: 실패 케이스 분석 ===")
display(df5)

### 관찰 및 결론

| 케이스 | 실패 원인 분류 | 개선 방향 |
|--------|-------------|----------|
| 1 (매출 계산) | LLM Hallucination | 계산 금지 system prompt 추가 |
| 2 (SNS 발언) | 문서 범위 초과 | 거부 응답 강화 |
| 3 (배터리 비교) | Chunk 분산 | chunk_size 확대 or k 증가 |
| 4 (모호한 질문) | Embedding 의미 왜곡 | 질문 재작성(실험 6-b)으로 개선 |
| 5 (주가 전망) | LLM Hallucination | 투기적 예측 금지 프롬프트 |

**실패 원인 중 가장 많았던 것**: *(실행 후 작성)*

**결론**: *(실행 후 직접 작성)*

---

## 실험 6: RAG 개선 시도

### 가설
> 세 가지 방법으로 기본 RAG의 품질을 개선할 수 있다:
> 1. **System Prompt 강화**: 거짓말 금지, 출처 명시 규칙 → hallucination 억제
> 2. **질문 재작성 (Query Rewriting)**: 모호한 질문을 구체화 → 검색 정확도 향상
> 3. **Re-ranking**: 검색 결과를 재정렬 → 컨텍스트 품질 향상 (❌ 범위 초과)

### 📚 참고 자료
| 파일 | 관련 섹션 |
|------|----------|
| `PRJ01_W2_002_Simple_RAG_Pipeline.ipynb` | `### Step 2: Retrieval and generation` › `` `5. 검색 및 생성` `` |
| `PRJ01_W2_007_Retriever.ipynb` | `` `(3) RAG 프롬프트 구성` ``, `` `(4) RAG 체인 구성` `` |
| `PRJ01_W1_003_Langchain_Components.ipynb` | `# 3. 프롬프트 템플릿 (Prompt Template)`, `# 4. 출력 파서 (Output Parser)` |
| `PRJ01_W1_004_LangSmith_LCEL.ipynb` | `# LCEL` › `1) Prompt + LLM`, `2) Prompt + LLM + Output Parser`, `3) RunnablePassthrough` |

### 6-a: System Prompt 강화

> 📚 `PRJ01_W2_002_Simple_RAG_Pipeline.ipynb` — `` `5. 검색 및 생성` `` (system prompt 포함 RAG 체인 구현)  
> 📚 `PRJ01_W2_007_Retriever.ipynb` — `` `(3) RAG 프롬프트 구성` ``, `` `(4) RAG 체인 구성` ``

In [ ]:
vs_exp6, _ = build_chroma(raw_docs, chunk_size=500, collection_name="exp6_improve")

# ── 기본 프롬프트 ──
BASIC_PROMPT = "컨텍스트를 기반으로 질문에 답하세요.\n\n컨텍스트:\n{context}"

# ── 강화 프롬프트 ──
ENHANCED_PROMPT = """당신은 테슬라와 리비안 문서를 분석하는 전문 AI 어시스턴트입니다.

[답변 규칙]
1. 반드시 아래 컨텍스트의 정보만 사용하세요.
2. 컨텍스트에 없는 정보는 "제공된 문서에서 확인되지 않습니다"라고 명확히 밝히세요.
3. 수치나 날짜를 언급할 때는 어느 회사(테슬라/리비안) 정보인지 명시하세요.
4. 추측, 예측, 수치 계산을 하지 마세요.
5. 답변은 정확하고 간결하게 작성하세요.

[컨텍스트]
{context}"""

chain_basic = build_rag_chain(vs_exp6, llm, k=3, system_prompt=BASIC_PROMPT)
chain_enhanced = build_rag_chain(vs_exp6, llm, k=3, system_prompt=ENHANCED_PROMPT)

# 실험 5 실패 케이스 재시도 + 일반 질문
compare_qs = [
    "테슬라의 주요 전기차 모델을 소개해주세요.",           # 일반 질문
    "테슬라와 리비안의 2024년 매출 합계를 계산해주세요.",  # 실패 케이스 1 재시도
    "이 회사들의 향후 주가 전망은 어떻게 되나요?",         # 실패 케이스 5 재시도
]

rows6a = []
for q in compare_qs:
    r_basic = chain_basic.invoke(q)
    r_enhanced = chain_enhanced.invoke(q)
    rows6a.append({"질문": q, "기본 프롬프트": r_basic, "강화 프롬프트": r_enhanced})

print("=== 6-a: System Prompt 강화 비교 ===")
for row in rows6a:
    print(f"\nQ: {row['질문']}")
    print(f"  [기본] {row['기본 프롬프트'][:200]}...")
    print(f"  [강화] {row['강화 프롬프트'][:200]}...")

print("\n✅ 6-a 완료")

In [ ]:
df6a = pd.DataFrame([{
    "질문 (요약)": r["질문"][:40] + "...",
    "기본 프롬프트 (처음 100자)": r["기본 프롬프트"][:100] + "...",
    "강화 프롬프트 (처음 100자)": r["강화 프롬프트"][:100] + "...",
} for r in rows6a])

print("=== 6-a 결과 요약 ===")
display(df6a)

### 6-b: 질문 재작성 (Query Rewriting)

모호하거나 짧은 질문을 LLM이 검색에 최적화된 질문으로 재작성 후 RAG에 투입.  
**흐름**: 원본 질문 → LLM 재작성 → 벡터 검색 → 최종 응답

> 📚 `PRJ01_W1_003_Langchain_Components.ipynb` — `# 3. 프롬프트 템플릿 (Prompt Template)`, `# 4. 출력 파서 (Output Parser)`  
> 📚 `PRJ01_W1_004_LangSmith_LCEL.ipynb` — `# LCEL` › `1) Prompt + LLM`, `2) Prompt + LLM + Output Parser`, `3) RunnablePassthrough`

In [ ]:
# 질문 재작성 체인
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 RAG 검색 최적화 전문가입니다.
사용자의 질문을 벡터 유사도 검색에 최적화된 명확한 질문으로 재작성하세요.

규칙:
- 모호하면 구체적으로 만드세요
- 핵심 키워드(제품명, 회사명, 기술 용어)가 포함되도록 하세요
- 원래 의도를 유지하세요
- 재작성된 질문만 출력하세요 (설명 없이)"""),
    ("human", "원본 질문: {question}"),
])

rewrite_chain = (
    rewrite_prompt
    | ChatOpenAI(model=DEFAULT_MODEL, temperature=0)
    | StrOutputParser()
)

# 실험 5 실패 케이스 재시도 (질문 재작성 적용)
ambiguous_qs = [
    "전기차",                    # 실패 케이스 4: 모호한 질문
    "리비안 거기 어때요?",        # 구어체 모호 표현
    "차 비교해줘",               # 주어 없는 짧은 질문
]

chain_rewrite_rag = build_rag_chain(vs_exp6, llm, k=3)

rows6b = []
print("=== 6-b: 질문 재작성 효과 ===")
for q in ambiguous_qs:
    rewritten = rewrite_chain.invoke({"question": q})

    r_original = chain_rewrite_rag.invoke(q)
    r_rewritten = chain_rewrite_rag.invoke(rewritten)

    print(f"\n원본    : {q}")
    print(f"재작성  : {rewritten}")
    print(f"원본 응답 (처음 150자): {r_original[:150]}...")
    print(f"재작성 응답 (처음 150자): {r_rewritten[:150]}...")

    rows6b.append({
        "원본 질문": q,
        "재작성 질문": rewritten,
        "원본 응답 길이(자)": len(r_original),
        "재작성 응답 길이(자)": len(r_rewritten),
    })

df6b = pd.DataFrame(rows6b)
print("\n=== 6-b 결과 요약 ===")
display(df6b)
print("\n✅ 6-b 완료")

### 6-c: Re-ranking ❌ 현재 강의 범위 초과

> **Re-ranking이란?**  
> 벡터 검색으로 top-K 문서를 가져온 후, 더 정밀한 **Cross-Encoder 모델**로 관련성을 재평가하여 순위를 재조정하는 기법.

| 단계 | 설명 |
|------|------|
| Bi-Encoder (기존 RAG) | 질문과 문서를 별도 인코딩 후 코사인 유사도 → 빠르지만 정밀도 낮음 |
| Cross-Encoder (Re-ranking) | 질문+문서를 함께 입력 → 관련성 점수 정밀 계산, 느리지만 정확 |

#### 구현 방법 (참고용 - 추가 설치 필요)
```python
# 방법 1: HuggingFace Cross-Encoder (pip install sentence-transformers)
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# from langchain.retrievers.document_compressors import CrossEncoderReranker
# from langchain.retrievers import ContextualCompressionRetriever
#
# cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
# compressor = CrossEncoderReranker(model=cross_encoder, top_n=3)
# base_retriever = vs.as_retriever(search_kwargs={"k": 10})  # 넓게 가져옴
# rerank_retriever = ContextualCompressionRetriever(
#     base_compressor=compressor,
#     base_retriever=base_retriever
# )

# 방법 2: Cohere Reranker (pip install langchain-cohere, Cohere API 키 필요)
# from langchain_cohere import CohereRerank
# compressor = CohereRerank(model="rerank-multilingual-v3.0", top_n=3)
```

#### 예상 효과
```
k=10으로 넓게 검색 → Cross-Encoder로 top-3 재선별
→ 일반 k=3보다 높은 컨텍스트 품질 기대
```

### 관찰 및 결론

| 개선 방법 | 적용 결과 | 예상 효과 | 구현 복잡도 | 추가 비용 |
|----------|---------|---------|------------|----------|
| 6-a System Prompt 강화 | ✅ 완료 | | 낮음 | 없음 |
| 6-b Query Rewriting | ✅ 완료 | | 중간 | LLM 호출 1회 추가 |
| 6-c Re-ranking | ❌ 범위 초과 | 높음 (예상) | 높음 | 추가 모델 필요 |

**결론**: *(실행 후 직접 작성)*

---

## 종합 결론 및 권장 설정

### 실험 결과 요약

| 실험 | 테스트 값 | 최적 설정 | 근거 |
|------|---------|---------|------|
| 1. Chunk Size | 200 / 500 / 1000 | *(작성)* | *(작성)* |
| 2. Top-k | 3 / 5 / 10 | *(작성)* | *(작성)* |
| 3. Temperature | 0 / 0.3 / 0.7 | *(작성)* | *(작성)* |
| 4. 벡터DB | FAISS / Chroma | *(작성)* | *(작성)* |
| 5. 실패 케이스 | - | 주요 원인: *(작성)* | - |
| 6. RAG 개선 | Prompt / Rewrite | *(작성)* | *(작성)* |

### 권장 기본 설정

```
chunk_size    : 500       (대부분의 경우 균형점)
chunk_overlap : 50        (10%)
k             : 3~5       (정밀도와 recall 균형)
temperature   : 0         (사실 기반 RAG에서는 낮게 유지)
vectorstore   : FAISS     (소규모 프로토타입)
               Chroma     (영속성 필요 또는 메타데이터 필터링 활용 시)
```

### 개선 우선순위 (비용 대비 효과)

```
1순위: System Prompt 강화       → 비용 0, 즉시 적용 가능
2순위: Query Rewriting          → LLM 호출 1회 추가, 모호한 질문에 효과적
3순위: Chunk Size 최적화        → 데이터 특성에 따라 튜닝
4순위: Re-ranking               → 고품질 필요 시, 추가 모델 설치 필요
```

### 발견한 실패 패턴과 해결책

*(실행 후 직접 작성)*